# Lesson 01b — API Integration & Streaming
**Domain:** E-commerce · Yelp Reviews  
**Dataset:** Yelp Open Dataset (Kaggle: `yelp-dataset`, `yelp_review.json` sample 10k)  
**Time estimate:** ~2 h

## Learning objectives
- OpenAI and Anthropic Python SDKs — key differences
- LM Studio as a local OpenAI-compatible server
- Streaming responses: yield tokens live, measure time-to-first-token (TTFT)
- Async batch processing with `asyncio.gather()`
- Rate limiting and retry with exponential backoff

In [ ]:
import sys, os, time, asyncio, random
sys.path.insert(0, "..")

import pandas as pd
from pathlib import Path
from openai import OpenAI, AsyncOpenAI

from llm_checker import check, hint, evaluate, progress

# ── Local LM Studio client ─────────────────────────────────────────
local_client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
async_local   = AsyncOpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
LOCAL_MODEL   = "local-model"

print("✅ Setup complete")
print("ℹ️  Make sure LM Studio is running at http://localhost:1234")

## Load Yelp Reviews

In [ ]:
YELP_PATH = Path("../data/yelp/yelp_review_sample.csv")

if YELP_PATH.exists():
    yelp_df = pd.read_csv(YELP_PATH)
else:
    # Fallback: create a small synthetic sample so the notebook runs without Kaggle
    print("⚠️  Yelp dataset not found at", YELP_PATH)
    print("   Using synthetic fallback reviews for exercises.")
    yelp_df = pd.DataFrame({
        "text": [
            "Absolutely loved this place! The food was incredible and service was fast.",
            "Terrible experience. Waited 45 minutes and the food was cold when it arrived.",
            "Decent spot. Nothing special but gets the job done for a quick lunch.",
            "Best pizza in town, hands down. Will definitely be back every week!",
            "The place was dirty and the staff seemed annoyed to be there. Never again.",
        ] * 4,
        "stars": [5, 1, 3, 5, 1] * 4,
    })

print(f"✅ {len(yelp_df)} reviews loaded")
yelp_df.head(3)

---
## 🔵 EXAMPLE — Ex 01b-1: Sync vs streaming call

Classify a Yelp review as `positive` / `neutral` / `negative` using both **sync** and
**streaming** modes. Measure time-to-first-token (TTFT) for the streaming call.

In [ ]:
# ── Ex 01b-1: Sync vs streaming (EXAMPLE) ────────────────────────
review_text = yelp_df["text"].iloc[0]
CLASSIFY_SYSTEM = "Classify the sentiment of this review as exactly one word: positive, neutral, or negative."

# ── Sync call ─────────────────────────────────────────────────────
t_sync_start = time.perf_counter()
sync_resp = local_client.chat.completions.create(
    model=LOCAL_MODEL,
    messages=[
        {"role": "system", "content": CLASSIFY_SYSTEM},
        {"role": "user",   "content": review_text},
    ],
    max_tokens=10,
)
sync_time  = time.perf_counter() - t_sync_start
sync_label = sync_resp.choices[0].message.content.strip().lower()

# ── Streaming call with TTFT measurement ──────────────────────────
t_stream_start = time.perf_counter()
stream = local_client.chat.completions.create(
    model=LOCAL_MODEL,
    messages=[
        {"role": "system", "content": CLASSIFY_SYSTEM},
        {"role": "user",   "content": review_text},
    ],
    max_tokens=10,
    stream=True,
)

tokens_received = []
ttft = None
for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    if delta and ttft is None:
        ttft = (time.perf_counter() - t_stream_start) * 1000  # ms
    tokens_received.append(delta)
    print(delta, end="", flush=True)

stream_label = "".join(tokens_received).strip().lower()
print()

print(f"\nSync  label : {sync_label!r}   ({sync_time*1000:.0f} ms total)")
print(f"Stream label: {stream_label!r}  (TTFT = {ttft:.0f} ms)")

check([
    (ttft is not None,             "TTFT measured and printed in ms"),
    (ttft < 5000,                  "TTFT < 5 s (LM Studio is responsive)"),
    (sync_label == stream_label,   "Sync and streaming produce same label"),
], exercise_id="01b-1")

---
## 🟡 EXERCISE — Ex 01b-2: Async batch sentiment classification

Write `async_classify_sentiment(reviews: list[str]) -> list[str]` classifying Yelp reviews
in parallel using `asyncio.gather()`.  
Run on 20 reviews. Measure speedup vs sequential.

**Auto-check verifies:**
- All 20 reviews classified without exception
- Parallel time < sequential time
- Speedup ratio printed (expect ≥ 2×)

In [ ]:
# ── Your implementation ──────────────────────────────────────────────

async def _classify_one(client: AsyncOpenAI, text: str) -> str:
    resp = await client.chat.completions.create(
        model=LOCAL_MODEL,
        messages=[
            {"role": "system", "content": "Classify sentiment as: positive, neutral, or negative. One word only."},
            {"role": "user",   "content": text[:500]},
        ],
        max_tokens=5,
    )
    return resp.choices[0].message.content.strip().lower()


async def async_classify_sentiment(reviews: list) -> list:
    # TODO: use asyncio.gather to classify all reviews in parallel
    raise NotImplementedError("Implement async_classify_sentiment()")

In [ ]:
# ── Auto-check ───────────────────────────────────────────────────────
reviews_20 = yelp_df["text"].tolist()[:20]

async def run_checks():
    try:
        # Parallel
        t0 = time.perf_counter()
        labels_parallel = await async_classify_sentiment(reviews_20)
        t_parallel = time.perf_counter() - t0

        # Sequential (for comparison)
        t0 = time.perf_counter()
        labels_seq = [await _classify_one(async_local, r) for r in reviews_20]
        t_sequential = time.perf_counter() - t0

        speedup = t_sequential / max(t_parallel, 0.001)
        print(f"Parallel  : {t_parallel:.2f}s")
        print(f"Sequential: {t_sequential:.2f}s")
        print(f"Speedup   : {speedup:.1f}×")

        return check([
            (len(labels_parallel) == 20,  "All 20 reviews classified"),
            (all(isinstance(l, str) for l in labels_parallel), "All labels are strings"),
            (t_parallel < t_sequential,   "Parallel faster than sequential"),
            (speedup >= 1.5,              "Speedup ≥ 1.5×"),
        ], exercise_id="01b-2")
    except NotImplementedError:
        print("⚠️  Implement async_classify_sentiment() first.")

await run_checks()

---
## 🟡 EXERCISE — Ex 01b-3: Retry decorator with exponential backoff

Write a `@retry(max_attempts=3, base_delay=1.0)` decorator that catches
`openai.RateLimitError` and `openai.APIError`, waits `base_delay * 2^attempt` seconds,
and retries.  
Test with a mock that raises on the first 2 attempts.

**Auto-check verifies:**
- Decorator works on any callable
- After `max_attempts` failures, raises the last exception
- Function succeeds on attempt 3 → returns result
- Sleep delays are logged

In [ ]:
import functools, time as _time
from openai import RateLimitError

call_log: list = []   # track attempt counts for testing

def retry(max_attempts: int = 3, base_delay: float = 1.0):
    """Exponential-backoff retry decorator for OpenAI API errors."""
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            # TODO: implement retry logic
            raise NotImplementedError("Implement the retry decorator")
        return wrapper
    return decorator

In [ ]:
# ── Auto-check ───────────────────────────────────────────────────────
from unittest.mock import MagicMock, patch

call_count = 0

def make_flaky_fn(fail_times: int):
    """Returns a function that raises RateLimitError `fail_times` times then succeeds."""
    calls = {"n": 0}
    def flaky():
        calls["n"] += 1
        if calls["n"] <= fail_times:
            raise RateLimitError("rate limit", response=MagicMock(status_code=429), body={})
        return "success"
    return flaky

try:
    # Test 1: succeeds on attempt 3
    flaky = make_flaky_fn(fail_times=2)
    decorated = retry(max_attempts=3, base_delay=0.01)(flaky)  # tiny delay for testing

    with patch("time.sleep") as mock_sleep:
        result = decorated()

    sleep_calls = [call.args[0] for call in mock_sleep.call_args_list]

    # Test 2: raises after max_attempts
    always_fail = retry(max_attempts=3, base_delay=0.01)(make_flaky_fn(fail_times=99))
    exhausted = False
    try:
        with patch("time.sleep"):
            always_fail()
    except RateLimitError:
        exhausted = True

    check([
        (result == "success",       "Succeeds on attempt 3 → returns result"),
        (len(sleep_calls) == 2,     "Exactly 2 sleep calls before success"),
        (exhausted,                 "Raises after max_attempts exhausted"),
        (all(d > 0 for d in sleep_calls), "Sleep delays are positive"),
    ], exercise_id="01b-3")

except NotImplementedError:
    print("⚠️  Implement the retry decorator first.")

In [ ]:
# ── Optional: mentor feedback ─────────────────────────────────────────
# evaluate(retry, "Write a @retry decorator with exponential backoff ...", exercise_id="01b-3")

---
## Readiness checklist before moving to 01c
- [ ] LM Studio server is running and reachable at localhost:1234
- [ ] Streaming call prints tokens live and TTFT is measured
- [ ] `async_classify_sentiment()` achieves ≥ 2× speedup
- [ ] Retry decorator handles `RateLimitError` with correct delays

In [ ]:
progress()